In [2]:
# 1. Instalar e carregar os pacotes necessários
#install.packages(c("bibliometrix", "dplyr", "stringr"))
library(bibliometrix)
library(dplyr)
library(stringr)

Installing packages into ‘/home/silvio/R/x86_64-pc-linux-gnu-library/4.3’
(as ‘lib’ is unspecified)

Please note that our software is open source and available for use, distributed under the MIT license.
When it is used in a publication, we ask that authors properly cite the following reference:

Aria, M. & Cuccurullo, C. (2017) bibliometrix: An R-tool for comprehensive science mapping analysis,
                        Journal of Informetrics, 11(4), pp 959-975, Elsevier.

Failure to properly cite the software is considered a violation of the license.
                        
For information and bug reports:
                        - Take a look at https://www.bibliometrix.org
                        - Send an email to info@bibliometrix.org
                        - Write a post on https://github.com/massimoaria/bibliometrix/issues
                        
Help us to keep Bibliometrix and Biblioshiny free to download and use by contributing with a small donation to support our research

In [2]:
print(getwd())

[1] "/home/silvio/Projetos/TEMAC/bibliometrix"


In [4]:
# Carregar o arquivo Excel no caminho correto
arquivo <- file.path("..","ajustes_BD", "IC2I", "metadados_iccrts_v4.xlsx")

if (!file.exists(arquivo)) {
    stop(paste("Arquivo não encontrado:", arquivo))
}
#2. Carregar os dados do CSV (substitua pelo caminho correto, se necessário)
df_raw <- as.data.frame(readxl::read_excel(arquivo), stringsAsFactors = FALSE)

In [5]:

#df_raw <- read.csv("metadados_iccrts_v4.xlsx - Sheet1.csv", stringsAsFactors = FALSE)

# 3. Transformar e Mapear as colunas para o padrão Web of Science
M <- df_raw %>%
  mutate(
    # TI: Título (Title)
    TI = TITLE,
    
    # PY: Ano de Publicação (Publication Year)
    PY = as.numeric(YEAR),
    
    # SO: Nome da Fonte / Conferência (Source Title)
    SO = CONFERENCE,
    
    # AB: Resumo (Abstract)
    AB = ABSTRACT,
    
    # DE: Palavras-chave dos Autores (Author Keywords) - Convertendo para maiúsculo
    # Usei LLM_KEYWORDS, mas você pode trocar por KEYWORDS se preferir
    DE = toupper(LLM_KEYWORDS), 
    
    # AU: Autores (Authors)
    # Remove pontos e vírgulas, converte para maiúsculas
    AU = str_remove_all(AUTHORS_APA, "[,.]"),
    AU = toupper(AU),
    
    # CR: Referências Citadas (Cited References)
    # Substitui o "|" (pipe) por ";" (ponto-e-vírgula)
    CR = str_replace_all(REFERENCES, "\\|", ";"),
    
    # DT: Tipo de Documento
    DT = "CONFERENCE PAPER",
    
    # TC: Total de Citações (Zerar, já que o arquivo parece não ter essa métrica)
    TC = 0,
    
    # DB: Identificador da Base de Dados
    DB = "CUSTOM"
  ) %>%
  # Seleciona apenas as colunas formatadas (as "Tags" do Bibliometrix)
  select(TI, AU, PY, SO, AB, DE, CR, DT, TC, DB)



In [6]:
# 4. Ajustes finais do Data Frame
M <- as.data.frame(M)

# O bibliometrix precisa de identificadores únicos para cada linha (Short Reference - SR)
# Vamos extrair o primeiro nome do primeiro autor para montar o ID (Ex: TRENT S, 2015, CONFERENCE)
primeiro_autor <- word(M$AU, 1, sep = ";") %>% str_trim() %>% word(1)
M$SR <- paste(primeiro_autor, M$PY, M$SO, sep = ", ")

# Evita erros se houver artigos do mesmo autor no mesmo ano (adiciona .1, .2)
M$SR <- make.unique(M$SR) 
rownames(M) <- M$SR

# 5. Testando a conversão com a Análise Bibliométrica
# Se tudo estiver correto, a função abaixo deve gerar os resultados sem erros
resultados <- biblioAnalysis(M, sep = ";")

# Exibe o resumo geral
summary(resultados)



MAIN INFORMATION ABOUT DATA

 Timespan                              2016 : 2025 
 Sources (Journals, Books, etc)        10 
 Documents                             488 
 Annual Growth Rate %                  3.25 
 Document Average Age                  5.37 
 Average citations per doc             0 
 Average citations per year per doc    0 
 References                            11943 
 
DOCUMENT TYPES                     
 conference paper      488 
 
DOCUMENT CONTENTS
 Keywords Plus (ID)                    0 
 Author's Keywords (DE)                2303 
 
AUTHORS
 Authors                               871 
 Author Appearances                    1542 
 Authors of single-authored docs       65 
 
AUTHORS COLLABORATION
 Single-authored docs                  110 
 Documents per Author                  0.56 
 Co-Authors per Doc                    3.16 
 International co-authorships %        0 
 

Annual Scientific Production

 Year    Articles
    2016       36
    2017       51
    2018

In [7]:
M_ICCRTS <-M

In [12]:
save(M_ICCRTS, file = "../M_iccrts.rdata")

In [3]:
biblioshiny()

Loading required package: shiny




Listening on http://127.0.0.1:6048

